# 04 · SpaceX Mission-Level SQL Analysis
**Portable SQLite edition**

The original IBM SQL exercise used a remote IBM Db2 database and included connection credentials. This modern version requires **no database account, secrets, SQL extension, or legacy SQLAlchemy pin**. It loads the separate IBM mission-level `Spacex.csv` teaching dataset into an in-memory SQLite database and executes the original ten analytical tasks using portable SQL.

**Important:** This mission-level dataset has different columns and coverage from the 90-row Falcon 9 recovery dataset. Do not compare its totals to Notebook 02 without aligning definitions. Original IBM exercise preserved in `original-coursework/`.

## 1. Acquire the mission-level SQL dataset

In [1]:
from pathlib import Path
import io
import re
import sqlite3
import pandas as pd
import requests
ROOT=Path.cwd()
RAW=ROOT/'data'/'raw'; PROCESSED=ROOT/'data'/'processed'
RAW.mkdir(parents=True,exist_ok=True);PROCESSED.mkdir(parents=True,exist_ok=True)
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
)
RAW_FILE=RAW/'ibm_spacex_sql_missions.csv'
if not RAW_FILE.exists():
    response=requests.get(DATA_URL,timeout=30)
    response.raise_for_status()
    RAW_FILE.write_bytes(response.content)
missions=pd.read_csv(RAW_FILE)
print('Original columns:',missions.columns.tolist())
print('Shape:',missions.shape)
display(missions.head())

Original columns: ['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']
Shape: (101, 10)


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## 2. Normalise column names and create a local SQL table

IBM versions of this dataset have used inconsistent punctuation in column names. We normalise names for portable SQL while retaining the original CSV on disk.

In [2]:
def clean_name(name):
    return re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]+','_',str(name))).strip('_').lower()
missions.columns=[clean_name(c) for c in missions.columns]
print('SQL columns:',missions.columns.tolist())
if missions.columns.duplicated().any():
    raise ValueError('Column normalisation produced duplicate names')
conn=sqlite3.connect(':memory:')
missions.to_sql('missions',conn,index=False,if_exists='replace')
def query(sql):
    return pd.read_sql_query(sql,conn)

def column_for(*candidates):
    for candidate in candidates:
        if candidate in missions.columns: return candidate
    raise KeyError(f'Expected one of {candidates}; available: {missions.columns.tolist()}')
site=column_for('launch_site')
payload=column_for('payload_mass_kg','payload_mass_kg_')
booster=column_for('booster_version')
customer=column_for('customer')
landing=column_for('landing_outcome')
mission=column_for('mission_outcome')
date_col=column_for('date')
print('Validated mission-level SQL schema')

SQL columns: ['date', 'time_utc', 'booster_version', 'launch_site', 'payload', 'payload_mass_kg', 'orbit', 'customer', 'mission_outcome', 'landing_outcome']
Validated mission-level SQL schema


## 3. Original ten SQL investigations

In [3]:
tasks = {
    '01 · Unique launch sites': 'SELECT DISTINCT "{site}" AS launch_site FROM missions ORDER BY launch_site',
    '02 · Five CCA launches': 'SELECT * FROM missions WHERE "{site}" LIKE \'CCA%\' LIMIT 5',
    '03 · NASA CRS payload mass': 'SELECT SUM("{payload}") AS total_payload_kg FROM missions WHERE "{customer}" = \'NASA (CRS)\'',
    '04 · F9 v1.1 mean payload': 'SELECT AVG("{payload}") AS mean_payload_kg FROM missions WHERE "{booster}" = \'F9 v1.1\'',
    '05 · First successful ground landing': 'SELECT MIN("{date_col}") AS first_ground_success FROM missions WHERE "{landing}" = \'Success (ground pad)\'',
    '06 · Drone-ship success, 4–6 tonnes': 'SELECT "{booster}" AS booster, "{payload}" AS payload_kg FROM missions WHERE "{landing}" = \'Success (drone ship)\' AND "{payload}" > 4000 AND "{payload}" < 6000',
    '07 · Mission outcome frequencies': 'SELECT "{mission}" AS mission_outcome, COUNT(*) AS launches FROM missions GROUP BY "{mission}" ORDER BY launches DESC',
    '08 · Maximum payload booster': 'SELECT "{booster}" AS booster, "{payload}" AS payload_kg FROM missions WHERE "{payload}" = (SELECT MAX("{payload}") FROM missions)',
    '09 · Failed drone-ship landings in 2015': 'SELECT "{landing}" AS landing_outcome, "{booster}" AS booster, "{site}" AS launch_site FROM missions WHERE "{landing}" = \'Failure (drone ship)\' AND SUBSTR("{date_col}",1,4) = \'2015\'',
    '10 · Landing outcome rankings, 2010–2017': 'SELECT "{landing}" AS landing_outcome, COUNT(*) AS landing_count FROM missions WHERE "{date_col}" BETWEEN \'2010-06-04\' AND \'2017-03-20\' GROUP BY "{landing}" ORDER BY landing_count DESC',
}
variables = dict(site=site,payload=payload,booster=booster,customer=customer,landing=landing,mission=mission,date_col=date_col)
results={}
for title, template in tasks.items():
    sql = template.format(**variables)
    print('\n'+title)
    result=query(sql)
    results[title]=result
    display(result)
assert len(results)==10


01 · Unique launch sites


,launch_site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E



02 · Five CCA launches


,date,time_utc,booster_version,launch_site,payload,payload_mass_kg,orbit,customer,mission_outcome,landing_outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt



03 · NASA CRS payload mass


...

,booster,payload_kg
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600



09 · Failed drone-ship landings in 2015


,landing_outcome,booster,launch_site
0,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40



10 · Landing outcome rankings, 2010–2017


,landing_outcome,landing_count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


## 4. Data quality and export

In [4]:
print('Mission-level missing values:')
display(missions.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Duplicate full rows:',int(missions.duplicated().sum()))
output=PROCESSED/'spacex_sql_mission_dataset.csv'
missions.to_csv(output,index=False)
print('Saved:',output)
conn.close()

Mission-level missing values:


,missing
date,0
time_utc,0
booster_version,0
launch_site,0
payload,0
payload_mass_kg,0
orbit,0
customer,0
mission_outcome,0
landing_outcome,0


Duplicate full rows: 0
Saved: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\spacex_sql_mission_dataset.csv


## Limitations
- This dataset is **not** the same as the recovery-labelled Falcon 9 dataset; mission success and booster recovery are different outcomes.
- SQL task 9 uses `SUBSTR` rather than IBM Db2 `YEAR()`, and all queries use SQLite-compatible syntax.
- IBM's historical data may encode dates as strings; inspect formats before extending date comparisons.
- No secrets or connection credentials belong in notebooks or Git history.